## Imports and configurations

In [ ]:
import os
import torch
import numpy as np
from torch import nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.optim import AdamW
from torchvision import transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.utils import save_image
import medmnist
from medmnist import INFO
from pytorch_fid import fid_score
from torch.utils.data import DataLoader
import json
import time
from math import cos, pi

# ===============================
#           Config
# ===============================
class Config:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.data_flag = 'bloodmnist'
        self.batch_size = 64
        self.n_steps = 1000
        self.n_epochs = 200
        self.n_samples = 10000
        self.seeds = [42, 123, 456, 789, 999]
        
        # Diretórios de saída com timestamp para evitar sobreposição
        timestamp = time.strftime("%Y%m%d-%H%M%S")
        self.base_dir = f"./diffusion_run_{timestamp}"
        self.output_dir_real = os.path.join(self.base_dir, "real_samples")
        self.output_dir_gen = os.path.join(self.base_dir, "generated_samples")
        self.model_dir = os.path.join(self.base_dir, "models")
        self.log_dir = os.path.join(self.base_dir, "logs")
        
        # Parâmetros de modelo
        self.base_channels = 64
        self.channel_mults = [1, 2, 4, 4]
        self.time_emb_dim = 256
        self.dropout_rate = 0.1
        self.use_attention = True
        self.weight_decay = 1e-4
        self.learning_rate = 2e-4
        self.scheduler_type = 'wave'  # linear, wave, cosine
        
        # Configuração do dataset
        info = INFO[self.data_flag]
        self.DataClass = getattr(medmnist, info['python_class'])
        self.n_channels = info['n_channels']
        self.n_classes = len(info['label'])
        
        # Criar diretórios
        for dir_path in [self.output_dir_real, self.output_dir_gen, 
                         self.model_dir, self.log_dir]:
            os.makedirs(dir_path, exist_ok=True)
        
        # Salvar configuração
        self.save_config()
    
    def save_config(self):
        """Salva a configuração em um arquivo JSON"""
        config_dict = {k: v for k, v in self.__dict__.items() 
                      if not k.startswith('__') and not callable(v) and not isinstance(v, torch.device)}
        
        # Converter tipos não serializáveis
        config_dict['device'] = str(self.device)
        config_dict['DataClass'] = self.DataClass.__name__
        
        with open(os.path.join(self.base_dir, 'config.json'), 'w') as f:
            json.dump(config_dict, f, indent=2)

# verify cuda availability
if not torch.cuda.is_available():
    config = Config()
    print(config.device)
    raise RuntimeError("CUDA is not available. Please check your setup.")

## Data configuration

In [ ]:
def get_loader(config):
    transform = get_data_transform(config.augmentation)
    train_dataset = config.DataClass(split='train', transform=transform, download=True)
    loader = DataLoader(train_dataset, batch_size=config.batch_size, 
                       shuffle=True, num_workers=2, pin_memory=True)
    return loader

def get_data_transform():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomRotation(5),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

## Save samples

In [ ]:
def save_real_samples(config, loader):
    os.makedirs(config.output_dir_real, exist_ok=True)

    real_images = []
    for batch, _ in loader:
        real_images.append(batch)
        if len(real_images) * config.batch_size >= config.n_samples:
            break

    real_images = torch.cat(real_images)[:config.n_samples]

    for i in range(config.n_samples):
        save_image(real_images[i], os.path.join(config.output_dir_real, f"{i}.png"))
    
    print(f"Amostras reais salvas em {config.output_dir_real}")


## Embedding

In [ ]:
def wave_embedding(n_steps, dim):
    """Embedding sinusoidal melhorado com componente de onda"""
    position = torch.arange(n_steps).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, dim, 2) * -(np.log(10000.0) / dim))
    
    # Componente sinusoidal padrão
    pe = torch.zeros(n_steps, dim)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    
    # Adicionar componente de onda para enriquecer o embedding
    wave_factor = torch.sin(position / n_steps * pi) * 0.1
    pe = pe * (1.0 + wave_factor)
    
    return pe

## Blocks

In [ ]:
class AdvancedBlock(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None, residual=False, dropout=0.0):
        super().__init__()
        self.residual = residual
        
        if not mid_channels:
            mid_channels = out_channels
        
        # Primeira camada convolucional
        self.conv1 = nn.Conv2d(in_channels, mid_channels, 3, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(min(8, mid_channels), mid_channels)
        
        # Segunda camada convolucional
        self.conv2 = nn.Conv2d(mid_channels, out_channels, 3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(min(8, out_channels), out_channels)
        
        # Dropout para regularização
        self.dropout = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        
        # Conexão residual se necessário
        if self.residual and in_channels != out_channels:
            self.skip = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        # Caminho principal
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.gelu(h)
        h = self.dropout(h)
        
        h = self.conv2(h)
        h = self.norm2(h)
        
        # Conexão residual
        if self.residual:
            return F.gelu(h + self.skip(x))
        else:
            return h


class EnhancedAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.norm = nn.GroupNorm(min(8, channels), channels)
        
        # Projeções QKV unificadas para eficiência
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        
        # Projeção de saída
        self.proj = nn.Conv2d(channels, channels, 1)
        
        # Fator de escala para estabilidade
        self.scale = channels ** -0.5

    def forward(self, x):
        B, C, H, W = x.shape
        
        # Normalização
        h = self.norm(x)
        
        # Projeções QKV
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=1)
        
        # Reshape para multiplicação de matrizes
        q = q.reshape(B, C, -1).transpose(1, 2)  # B, HW, C
        k = k.reshape(B, C, -1)  # B, C, HW
        v = v.reshape(B, C, -1).transpose(1, 2)  # B, HW, C
        
        # Cálculo da atenção com escala
        attn = torch.softmax(torch.bmm(q, k) * self.scale, dim=-1)
        
        # Aplicação da atenção
        out = torch.bmm(attn, v).transpose(1, 2).reshape(B, C, H, W)
        
        # Projeção final com conexão residual
        return x + self.proj(out)




class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=256, dropout=0.0):
        super().__init__()
        
        # Upsampling
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        
        # Blocos convolucionais
        self.conv1 = AdvancedBlock(in_channels, in_channels, residual=True, dropout=dropout)
        self.conv2 = AdvancedBlock(in_channels, out_channels, in_channels // 2, dropout=dropout)
        
        # Camada de embedding temporal
        self.time_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_channels)
        )

    def forward(self, x, skip, t):
        # Upsampling
        x = self.up(x)
        
        # Ajustar dimensões se necessário
        diffY = skip.size()[2] - x.size()[2]
        diffX = skip.size()[3] - x.size()[3]
        
        if diffY > 0 or diffX > 0:
            x = F.pad(x, [diffX // 2, diffX - diffX // 2,
                          diffY // 2, diffY - diffY // 2])
        
        # Concatenar com skip connection
        x = torch.cat([skip, x], dim=1)
        
        # Convoluções
        x = self.conv1(x)
        x = self.conv2(x)
        
        # Adicionar informação temporal
        time_emb = self.time_layer(t)[:, :, None, None]
        time_emb = time_emb.expand(-1, -1, x.shape[-2], x.shape[-1])
        
        return x + time_emb

class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=256, dropout=0.0):
        super().__init__()
        
        # Downsampling
        self.pool = nn.MaxPool2d(2)
        
        # Blocos convolucionais
        self.conv1 = AdvancedBlock(in_channels, in_channels, residual=True, dropout=dropout)
        self.conv2 = AdvancedBlock(in_channels, out_channels, dropout=dropout)
        
        # Camada de embedding temporal
        self.time_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_channels)
        )

    def forward(self, x, t):
        # Downsampling e convoluções
        x = self.pool(x)
        x = self.conv1(x)
        x = self.conv2(x)
        
        # Adicionar informação temporal
        time_emb = self.time_layer(t)[:, :, None, None]
        time_emb = time_emb.expand(-1, -1, x.shape[-2], x.shape[-1])
        
        return x + time_emb


## UNet

In [ ]:
class BloodUNetPlus(nn.Module):
    def __init__(self, config):
        super().__init__()
        
        # Parâmetros
        self.device = config.device
        self.time_dim = config.time_emb_dim
        self.base_channels = config.base_channels
        self.channel_mults = config.channel_mults
        self.dropout = config.dropout_rate
        self.use_attention = config.use_attention
        
        # Embedding temporal
        self.time_embed = nn.Embedding(config.n_steps, self.time_dim)
        self.time_embed.weight.data = wave_embedding(config.n_steps, self.time_dim)
        self.time_embed.requires_grad_(False)
        
        # Calcular canais
        channels = [self.base_channels * m for m in self.channel_mults]
        
        # Camada inicial
        self.input_conv = AdvancedBlock(config.n_channels, channels[0], dropout=self.dropout)
        
        # Encoder
        self.down1 = DownBlock(channels[0], channels[1], self.time_dim, self.dropout)
        self.down2 = DownBlock(channels[1], channels[2], self.time_dim, self.dropout)
        self.down3 = DownBlock(channels[2], channels[3], self.time_dim, self.dropout)
        
        # Bottleneck
        self.bottleneck1 = AdvancedBlock(channels[3], channels[3]*2, dropout=self.dropout)
        self.bottleneck_attn = EnhancedAttention(channels[3]*2) if self.use_attention else nn.Identity()
        self.bottleneck2 = AdvancedBlock(channels[3]*2, channels[3], dropout=self.dropout)
        
        # Decoder
        self.up1 = UpBlock(channels[3]*2, channels[2], self.time_dim, self.dropout)
        self.up2 = UpBlock(channels[2]+channels[1], channels[1], self.time_dim, self.dropout)
        self.up3 = UpBlock(channels[1]+channels[0], channels[0], self.time_dim, self.dropout)
        
        # Camada de saída
        self.output_conv = nn.Conv2d(channels[0], config.n_channels, 3, padding=1)

    def pos_encoding(self, t):
        """Codificação posicional avançada para timesteps"""
        t = t.unsqueeze(-1).float()
        
        # Frequências
        inv_freq = 1.0 / (
            10000 ** (torch.arange(0, self.time_dim, 2, device=self.device).float() / self.time_dim)
        )
        
        # Componentes seno e cosseno
        pos_enc_a = torch.sin(t.repeat(1, self.time_dim // 2) * inv_freq)
        pos_enc_b = torch.cos(t.repeat(1, self.time_dim // 2) * inv_freq)
        
        # Concatenar
        pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=-1)
        
        return pos_enc

    def forward(self, x, t):
        # Embedding temporal
        t_emb = self.pos_encoding(t)
        
        # Encoder
        x0 = self.input_conv(x)
        x1 = self.down1(x0, t_emb)
        x2 = self.down2(x1, t_emb)
        x3 = self.down3(x2, t_emb)
        
        # Bottleneck
        x3 = self.bottleneck1(x3)
        x3 = self.bottleneck_attn(x3)
        x3 = self.bottleneck2(x3)
        
        # Decoder com skip connections
        x = self.up1(x3, x2, t_emb)
        x = self.up2(x, x1, t_emb)
        x = self.up3(x, x0, t_emb)
        
        # Saída
        return self.output_conv(x)

## DDPM

In [ ]:
class WaveDDPM:
    def __init__(self, network, config):
        self.network = network.to(config.device)
        self.config = config
        
        # Inicializar betas de acordo com o scheduler escolhido
        if config.scheduler_type == 'linear':
            self.betas = torch.linspace(1e-4, 0.02, config.n_steps).to(config.device)
        elif config.scheduler_type == 'cosine':
            self.betas = self._cosine_beta_schedule(config.n_steps).to(config.device)
        else:  # wave
            self.betas = self._wave_beta_schedule(config.n_steps).to(config.device)
        
        # Calcular alphas
        self.alphas = 1. - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        
        # Valores pré-calculados para sampling
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / self.alphas)
        
        # Para sampling acelerado
        self.posterior_variance = self.betas * (1. - self.alphas_cumprod_prev) / (1. - self.alphas_cumprod)
    
    # Método para calcular a programação de beta com base no cosseno
    def _cosine_beta_schedule(self, timesteps, s=0.008):
        steps = timesteps + 1
        x = torch.linspace(0, timesteps, steps)
        alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * np.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clip(betas, 0, 0.999)
    
    # Método para calcular a programação de beta com base em uma onda
    def _wave_beta_schedule(self, timesteps):
        # Combina linear com componente de onda
        base = torch.linspace(1e-4, 0.02, timesteps)
        wave = 0.005 * torch.sin(torch.linspace(0, 4*np.pi, timesteps))
        betas = base + wave
        return torch.clip(betas, 1e-4, 0.999)

    # Método para calcular o alpha_cumprod para um timestep específico
    def q_sample(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        
        sqrt_alpha_t = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_t = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
        
        return sqrt_alpha_t * x_start + sqrt_one_minus_alpha_t * noise, noise

    # Método para amostragem de um passo
    def p_sample(self, x, t, ema_model=None):
        model = ema_model if ema_model is not None else self.network
        model.eval()
        
        with torch.no_grad():
            # Predição do ruído
            predicted_noise = model(x, t)
            
            # Parâmetros para o passo atual
            alpha = self.alphas[t].view(-1, 1, 1, 1)
            alpha_cumprod = self.alphas_cumprod[t].view(-1, 1, 1, 1)
            beta = self.betas[t].view(-1, 1, 1, 1)
            
            # Caso especial para t=0
            if t[0] == 0:
                return (x - beta / torch.sqrt(1 - alpha_cumprod) * predicted_noise) / torch.sqrt(alpha)
            
            # Predição da imagem original
            pred_x0 = (x - torch.sqrt(1 - alpha_cumprod) * predicted_noise) / torch.sqrt(alpha_cumprod)
            pred_x0 = torch.clamp(pred_x0, -1, 1)
            
            # Média da distribuição posterior
            mean = (
                x * (alpha - alpha_cumprod) / torch.sqrt((1 - alpha_cumprod) * (1 - alpha)) +
                pred_x0 * torch.sqrt(alpha) * (1 - alpha_cumprod) / torch.sqrt((1 - alpha_cumprod) * (1 - alpha))
            )
            
            # Variância
            variance = self.posterior_variance[t].view(-1, 1, 1, 1)
            std = torch.sqrt(variance)
            
            # Amostragem
            return mean + std * torch.randn_like(x)
    
    # Método para amostragem acelerada
    def accelerated_sampling(self, ema_model=None, n_samples=16, steps=50):
        model = ema_model if ema_model is not None else self.network
        model.eval()
        
        # Selecionar timesteps para sampling acelerado
        if steps < self.config.n_steps:
            skip = self.config.n_steps // steps
            timesteps = list(range(0, self.config.n_steps, skip))[:steps]
            timesteps = sorted(timesteps, reverse=True)
        else:
            timesteps = list(range(self.config.n_steps - 1, -1, -1))
        
        # Iniciar com ruído
        x = torch.randn(n_samples, self.config.n_channels, 28, 28).to(self.config.device)
        
        # Processo de denoising
        with torch.no_grad():
            for i, timestep in enumerate(tqdm(timesteps, desc="Gerando amostras")):
                # Batch de timesteps
                t_batch = torch.full((n_samples,), timestep, device=self.config.device, dtype=torch.long)
                
                # Predição do ruído
                predicted_noise = model(x, t_batch)
                
                # Parâmetros para o passo atual
                alpha_cumprod_t = self.alphas_cumprod[timestep]
                
                # Próximo alpha_cumprod (ou 1.0 se for o último passo)
                alpha_cumprod_next = self.alphas_cumprod[timesteps[i+1]] if i < len(timesteps)-1 else torch.tensor(1.0).to(self.config.device)
                
                # Predição da imagem original
                pred_x0 = (x - torch.sqrt(1 - alpha_cumprod_t) * predicted_noise) / torch.sqrt(alpha_cumprod_t)
                pred_x0 = torch.clamp(pred_x0, -1, 1)
                
                # Direção para o próximo passo
                direction = torch.sqrt(1 - alpha_cumprod_next) * predicted_noise
                
                # Próximo x
                x = torch.sqrt(alpha_cumprod_next) * pred_x0 + direction
        
        # Normalizar para [0, 1]
        x = (x.clamp(-1, 1) + 1) / 2
        
        return x

## Treino

In [ ]:
def train_ddpm(model, loader, config, seed):
    torch.manual_seed(seed)
    
    # Otimizador e scheduler
    optimizer = AdamW(
        model.network.parameters(), 
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )
    scheduler = CosineAnnealingLR(optimizer, T_max=config.n_epochs)
    
    # Criar modelo EMA
    ema_network = BloodUNetPlus(config).to(config.device)
    ema_network.load_state_dict(model.network.state_dict())
    ema_decay = 0.995
    
    # Histórico de treino
    train_losses = []
    
    for epoch in range(config.n_epochs):
        model.network.train()
        epoch_losses = []
        
        for x0, _ in tqdm(loader, desc=f"Epoch {epoch+1}/{config.n_epochs}"):
            x0 = x0.to(config.device)
            
            # Selecionar timesteps aleatórios
            t = torch.randint(0, config.n_steps, (x0.size(0),), device=config.device).long()
            
            # Adicionar ruído
            x_noisy, noise = model.q_sample(x0, t)
            
            # Predição do ruído
            noise_pred = model.network(x_noisy, t)
            
            # Calcular perda
            loss = F.mse_loss(noise, noise_pred)
            
            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Atualizar EMA
            with torch.no_grad():
                for p, ema_p in zip(model.network.parameters(), ema_network.parameters()):
                    ema_p.data.mul_(ema_decay).add_(p.data, alpha=1 - ema_decay)
            
            epoch_losses.append(loss.item())
        
        # Média das perdas da época
        avg_loss = sum(epoch_losses) / len(epoch_losses)
        train_losses.append(avg_loss)
        
        print(f"Epoch [{epoch+1}/{config.n_epochs}], Loss: {avg_loss:.4f}")
        
        # Atualizar learning rate
        scheduler.step()
        
        # Salvar modelo a cada 10 épocas
        if (epoch + 1) % 10 == 0:
            torch.save({
                'model': model.network.state_dict(),
                'ema_model': ema_network.state_dict(),
                'optimizer': optimizer.state_dict(),
                'epoch': epoch,
                'loss': avg_loss
            }, os.path.join(config.model_dir, f"model_epoch_{epoch+1}.pt"))
    
    # Salvar modelo final
    torch.save({
        'model': model.network.state_dict(),
        'ema_model': ema_network.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': config.n_epochs,
        'loss': avg_loss
    }, os.path.join(config.model_dir, "model_final.pt"))
    
    # Atribuir o modelo EMA
    model.ema_network = ema_network
    
    return train_losses

## Generate Samples

In [ ]:
def generate_and_save_samples(model, config):
    output_dir = config.output_dir_gen
    os.makedirs(output_dir, exist_ok=True)

    # Usar o modelo EMA para geração
    if hasattr(model, 'ema_network'):
        print("Usando modelo EMA para geração de amostras")
        ema_model = model.ema_network
    else:
        print("Modelo EMA não encontrado, usando modelo padrão")
        ema_model = model.network
    
    # Gerar amostras com sampling acelerado
    batch_size = 100
    n = config.n_samples
    
    for start in tqdm(range(0, n, batch_size), desc="Gerando lotes de amostras"):
        end = min(start + batch_size, n)
        current_batch = end - start
        
        # Gerar amostras
        samples = model.accelerated_sampling(
            ema_model=ema_model,
            n_samples=current_batch,
            steps=50  # Usar menos passos para aceleração
        )
        
        # Salvar amostras
        for i in range(current_batch):
            save_image(samples[i], os.path.join(output_dir, f"{start + i}.png"))
    
    print(f"Amostras geradas salvas em {output_dir}")

## MAIN

In [ ]:
def main():
    # Inicializar configuração
    config = Config()
    print(f"Dispositivo: {config.device}")
    
    # Carregar dados
    loader = get_loader(config)
    
    # Salvar amostras reais
    print("Salvando amostras reais...")
    save_real_samples(config, loader)
    
    # Executar experimentos com diferentes seeds
    for i, seed in enumerate(config.seeds):
        print(f"\nExecutando experimento {i+1} com seed {seed}")
        
        # Inicializar modelo
        network = BloodUNetPlus(config)
        model = WaveDDPM(network, config)
        
        # Treinar modelo
        print("Treinando modelo...")
        train_losses = train_ddpm(model, loader, config, seed)
        
        # Gerar e salvar amostras
        print("Gerando amostras...")
        generate_and_save_samples(model, config)
        
        # Calcular FID (opcional)
        try:
            print("Calculando FID score...")
            fid = fid_score.calculate_fid_given_paths(
                [config.output_dir_real, config.output_dir_gen],
                batch_size=50,
                device=config.device,
                dims=2048
            )
            print(f"FID Score: {fid:.4f}")
            
            # Salvar resultado do FID
            with open(os.path.join(config.base_dir, f"fid_score_seed_{seed}.txt"), "w") as f:
                f.write(f"FID Score: {fid:.4f}")
        except Exception as e:
            print(f"Erro ao calcular FID: {e}")
    
    print("\nTodos os experimentos concluídos!")


if __name__ == "__main__":
    main()
